# Step 2: Initial Data Analysis

**Project:** GlobalPartners Business Insights Pipeline
**Prepared by:** Polat
**Report:** `docs/02_data_analysis/Step2_Initial_Data_Analysis.md`

This notebook is the evidence behind the Step 2 report. Each section answers one
question, and the report's findings reference these sections.

**How to run:** Kernel → Restart & Run All. Every cell must run top to bottom
without errors before the notebook is committed.

**Conventions** (see `CLAUDE.md`):
- Raw CSVs are read as strings (`dtype=str`, `keep_default_na=False`) so nothing is silently converted.
- Column names are normalized to lowercase snake_case right after reading.
- Date formats are explicit: `creation_time_utc` = ISO 8601, `date_key` = `%d-%m-%Y`.
- All years are profiled; the 2023 scope is measured separately in section 11.

## 1. Load the data

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()
)
RAW_DIR = PROJECT_ROOT / "data" / "raw"

LINE_KEY = ["order_id", "lineitem_id"]  # grain of order_items, join key to options
BUSINESS_TZ = "America/New_York"        # assumption pending SME Q10


def load(name: str) -> pd.DataFrame:
    """Read a raw CSV as strings and normalize column names."""
    df = pd.read_csv(RAW_DIR / f"{name}.csv", dtype=str, keep_default_na=False)
    df.columns = df.columns.str.strip().str.lower()
    return df


items = load("order_items")
options = load("order_item_options")
dates = load("date_dim")

print("order_items       :", items.shape)
print("order_item_options:", options.shape)
print("date_dim          :", dates.shape)

order_items       : (203519, 13)
order_item_options: (193017, 6)
date_dim          : (365, 8)


### Typed copies

The raw frames stay as strings. These typed copies are used for numeric and date
analysis. Booleans are stored as `"TRUE"`/`"FALSE"` strings (finding 18).

In [2]:
items_n = items.assign(
    item_price=pd.to_numeric(items["item_price"]),
    item_quantity=pd.to_numeric(items["item_quantity"]),
    is_loyalty=items["is_loyalty"].map({"TRUE": True, "FALSE": False}),
    creation_time_utc=pd.to_datetime(items["creation_time_utc"], format="ISO8601", utc=True),
)
options_n = options.assign(
    option_price=pd.to_numeric(options["option_price"]),
    option_quantity=pd.to_numeric(options["option_quantity"]),
)
dates_n = dates.assign(date_key=pd.to_datetime(dates["date_key"], format="%d-%m-%Y"))

print("unmapped is_loyalty values:", items_n["is_loyalty"].isna().sum())
print(items_n.dtypes)

unmapped is_loyalty values: 0
app_name                            object
restaurant_id                       object
creation_time_utc      datetime64[ns, UTC]
order_id                            object
user_id                             object
printed_card_number                 object
is_loyalty                            bool
currency                            object
lineitem_id                         object
item_category                       object
item_name                           object
item_price                         float64
item_quantity                        int64
dtype: object


## 2. Distinct values per column

In [3]:
tables = {"order_items": items, "order_item_options": options, "date_dim": dates}

for table_name, df in tables.items():
    print(f"\n====== {table_name} ======")
    for col in df.columns:
        distinct = df[col].nunique()
        top5 = df[col].value_counts().head(5).to_dict()
        print(f"{col:<22} distinct={distinct:<8} top5={top5}")


====== order_items ======
app_name               distinct=3        top5={'Alltown Fresh': 201423, 'Alltown Neighborhood Perks': 1270, 'Alltown Fresh - DEVELOPMENT': 826}
restaurant_id          distinct=28       top5={'5fece84aadb0d51509c36f22': 32459, '6054db3495b70198148b456f': 23135, '5e7e35ec902ad5ac017b242a': 21416, '5f6a6c1537ab46bd38e9df75': 18888, '62f2ce9824813746ce6f5140': 12571}


creation_time_utc      distinct=131328   top5={'2021-02-08T12:16:06.916Z': 61, '2020-12-09T15:11:43.976Z': 27, '2022-05-03T14:14:56.527Z': 26, '2021-12-07T16:23:30.917Z': 25, '2022-04-27T13:51:27.864Z': 23}
order_id               distinct=131328   top5={'60212b86505ee90c1e92ec74': 61, '5fd0e92f4f5ee9301fda8df8': 27, '627138e08ae3be97e9047d77': 26, '61af8a82ffd125453e62ae37': 25, '62694a5fcb0036591d459146': 23}
user_id                distinct=20175    top5={'': 17808, '5f1b00e5535ee93e0cb768e7': 2538, '5ece77fe902ad501337b23fd': 2454, '609d681462e498b356e72a6d': 1133, '5eac88d6902ad598127b240b': 1046}
printed_card_number    distinct=5867     top5={'': 157435, '678011308999': 404, '678011390161': 325, '678022603958': 236, '678026969900': 227}
is_loyalty             distinct=2        top5={'FALSE': 157435, 'TRUE': 46084}
currency               distinct=1        top5={'USD': 203519}
lineitem_id            distinct=203519   top5={'64086b84d5fbd8718a0717e6': 1, '61e32046f546cc112a68cde4': 1,

lineitem_id            distinct=102712   top5={'5ed7ab59505ee96c0f7b23c8': 152, '5ed7ab55902ad5e7287b23cf': 148, '5ed7b479902ad5dd377b23e2': 148, '5f857286505ee9bd0607e190': 144, '5f85ae374f5ee95c647b23d7': 144}
option_group_name      distinct=133      top5={'Milk Options': 25841, 'Bacon Egg And Cheese Options': 14760, 'Breakfast Burrito Options': 12219, 'Chipotle Turkey Options': 9813, 'Smoothie Add-ons': 6815}
option_name            distinct=637      top5={'Add Smokehouse Bacon': 6781, 'Add Gluten Free Bread': 5832, 'Add Almond Milk': 5598, 'Add Whole Milk': 5483, 'Add Avocado': 5375}
option_price           distinct=16       top5={'0': 127980, '1': 37988, '2': 19619, '0.6': 2670, '0.75': 1565}
option_quantity        distinct=1        top5={'1': 193017}

====== date_dim ======
date_key               distinct=365      top5={'01-01-2023': 1, '09-09-2023': 1, '07-09-2023': 1, '06-09-2023': 1, '05-09-2023': 1}
year                   distinct=1        top5={'2023': 365}
month              

## 3. Dataset profile

In [4]:
orders = items_n.groupby("order_id").size()
customers = items_n.loc[items_n["user_id"] != "", "user_id"]

print(f"line items            : {len(items_n):,}")
print(f"orders                : {orders.size:,}")
print(f"line items per order  : avg {orders.mean():.2f}, max {orders.max()}")
print(f"identified customers  : {customers.nunique():,}")
print(f"blank user_id rows    : {(items_n['user_id'] == '').sum():,} "
      f"({(items_n['user_id'] == '').mean():.1%})")
print(f"locations             : {items_n['restaurant_id'].nunique()}")
print(f"apps                  : {items_n['app_name'].nunique()}")
print(f"categories / items    : {items_n['item_category'].nunique()} / {items_n['item_name'].nunique()}")
print(f"currencies            : {items_n['currency'].unique().tolist()}")
print(f"lineitem_id unique    : {items_n['lineitem_id'].is_unique}")
print(f"order_id+lineitem_id unique: {not items_n.duplicated(LINE_KEY).any()}")
print(f"timestamps per order  : {items_n.groupby('order_id')['creation_time_utc'].nunique().max()}")
print(f"order date range (UTC): {items_n['creation_time_utc'].min()} -> {items_n['creation_time_utc'].max()}")
print(f"date_dim range        : {dates_n['date_key'].min().date()} -> {dates_n['date_key'].max().date()}")

line items            : 203,519
orders                : 131,328
line items per order  : avg 1.55, max 61
identified customers  : 20,174
blank user_id rows    : 17,808 (8.8%)
locations             : 28
apps                  : 3
categories / items    : 46 / 432
currencies            : ['USD']
lineitem_id unique    : True
order_id+lineitem_id unique: True


timestamps per order  : 1
order date range (UTC): 2020-04-21 18:56:44.244000+00:00 -> 2024-02-21 02:52:35.366000+00:00
date_dim range        : 2023-01-01 -> 2023-12-31


## 4. Numeric ranges and the loyalty crosstab

Checks for negative or zero values, the full set of option prices (the doc says
negative = discount), and whether a blank `printed_card_number` always means a
non-loyalty order (finding 7).

In [5]:
print(items_n[["item_price", "item_quantity"]].describe())
print("item_price <= 0   :", (items_n["item_price"] <= 0).sum())
print("item_quantity <= 0:", (items_n["item_quantity"] <= 0).sum())

print("\noption_quantity values  :", sorted(options_n["option_quantity"].unique().tolist()))
print("all option_price values :", sorted(options_n["option_price"].unique().tolist()))
print("negative option_price   :", (options_n["option_price"] < 0).sum())

print()
print(pd.crosstab(
    items["is_loyalty"],
    items["printed_card_number"] == "",
    rownames=["is_loyalty"],
    colnames=["card_blank"],
))

          item_price  item_quantity
count  203519.000000  203519.000000
mean        8.739889       1.117768
std        19.568335       2.313234
min         0.000000       0.000000
25%         5.890000       1.000000
50%         8.000000       1.000000
75%        10.000000       1.000000
max      5000.000000     500.000000
item_price <= 0   : 156
item_quantity <= 0: 1

option_quantity values  : [1]
all option_price values : [0.0, 0.05, 0.25, 0.5, 0.6, 0.75, 1.0, 1.25, 1.5, 1.99, 2.0, 2.5, 3.0, 4.0, 6.0, 8.0]
negative option_price   : 0

card_blank  False   True 
is_loyalty               
FALSE           0  157435
TRUE        46084       0


## 5. Inspect the outliers

`price_x_qty` is a **hypothesis column**: it assumes `item_price` is a unit price,
as the doc says. The values it produces ($2.5M for one line of kimchi) are the
first sign that the assumption is wrong. Section 6 tests it properly.

In [6]:
items_n["price_x_qty"] = items_n["item_price"] * items_n["item_quantity"]

cols = ["app_name", "order_id", "user_id", "item_name", "item_price", "item_quantity", "price_x_qty"]

print("Top 10 by item_price x item_quantity:")
print(items_n.nlargest(10, "price_x_qty")[cols].to_string())

print("\nZero-price items (top 10 by count):")
print(items_n.loc[items_n["item_price"] == 0, "item_name"].value_counts().head(10))

Top 10 by item_price x item_quantity:
             app_name                  order_id                   user_id                   item_name  item_price  item_quantity  price_x_qty
118704  Alltown Fresh  5f3d1ba7505ee9374b7b23cc  5ee7285f505ee9f7490cf93d               Korean Kimchi     5000.00            500   2500000.00
111543  Alltown Fresh  5ed7972a4f5ee90547011c03  5ecf9eda505ee9682b445912     Meet Your Matcha - 12oz     3500.00            500   1750000.00
126586  Alltown Fresh  615f4c9662e4983d105aeaa4                                    Chili Chicken Bowl     3567.00            300   1070100.00
161049  Alltown Fresh  615ca07661e498d660e720b9                                               B.L.A.T     2967.00            300    890100.00
111542  Alltown Fresh  5ed7972a4f5ee90547011c03  5ecf9eda505ee9682b445912           Espresso - Double     1125.00            500    562500.00
165682  Alltown Fresh  5ecf5b9b505ee9613c7b23dd  5ecf53924f5ee9de377b23ca                  Tuscan Sun     1917

## 6. Is `item_price` a unit price or a line total?

Each item's typical unit price is its median price on quantity-1 lines. Every
multi-quantity line is then checked against both interpretations.

In [7]:
single = items_n[items_n["item_quantity"] == 1]                       # rows with a single unit
reference_price = single.groupby("item_name")["item_price"].median()  # typical unit price per item

multi = items_n[items_n["item_quantity"] > 1].copy()                  # multi-quantity rows
multi["reference_price"] = multi["item_name"].map(reference_price)
multi["price_per_unit"] = multi["item_price"] / multi["item_quantity"]

looks_like_unit = (multi["item_price"] - multi["reference_price"]).abs() < 0.01      # item_price = unit price
looks_like_total = (multi["price_per_unit"] - multi["reference_price"]).abs() < 0.01  # item_price = line total

print("multi-quantity rows      :", len(multi))
print("item_price = unit price  :", looks_like_unit.sum())
print("item_price = line total  :", looks_like_total.sum())
print("neither                  :", (~looks_like_unit & ~looks_like_total).sum())

multi-quantity rows      : 13650
item_price = unit price  : 0
item_price = line total  : 5972
neither                  : 7678


### The "neither" rows

If `item_price` were a unit price, `price_per_unit / reference_price` would cluster
near 1/quantity (0.5 or lower). A median near 1.0 means normal price variation
across locations and years.

In [8]:
neither = multi[~looks_like_unit & ~looks_like_total]
print("no reference price:", neither["reference_price"].isna().sum())

ratio = (neither["price_per_unit"] / neither["reference_price"]).dropna()
print(ratio.describe())

no reference price: 16
count    7662.000000
mean        1.023115
std         0.222109
min         0.000000
25%         0.831791
50%         0.983306
75%         1.150250
max         3.428571
dtype: float64


**Conclusion:** `item_price` is the line total. Line revenue uses `item_price`
as-is and is never multiplied by quantity (finding 28).

## 7. How are options recorded?

The data has no order-total column, so option pricing can't be checked directly
against what the customer paid. There is an indirect test, which ties back to
finding 9.

`option_quantity` is always 1, so an option bought twice must appear as two rows.
If options were recorded per unit, a quantity-3 line would have "Add Bacon" three
times. If options appear once per line regardless of quantity, they're recorded
per line.

In [9]:
option_counts = (
    options.groupby([*LINE_KEY, "option_group_name", "option_name"])
    .size()
    .reset_index(name="times_recorded")
)

line_quantity = items_n[[*LINE_KEY, "item_quantity"]]
merged = option_counts.merge(line_quantity, on=LINE_KEY, how="left")

print("option groups with no matching line item:", merged["item_quantity"].isna().sum())
print()
print(pd.crosstab(
    merged["item_quantity"].clip(upper=5),
    merged["times_recorded"].clip(upper=5),
    rownames=["item_quantity (5 = 5+)"],
    colnames=["times_recorded (5 = 5+)"],
))

option groups with no matching line item: 28



times_recorded (5 = 5+)       1    2   3   4    5
item_quantity (5 = 5+)                           
1.0                      180058  223  55  11  305
2.0                        7090    8   2   1    2
3.0                        1145    1   0   0    0
4.0                         486    2   0   0    1
5.0                        1295    4   0   0    1


### Repeated identical options on quantity-1 lines (finding 9)

On a quantity-1 line, a repeated option can't be explained by per-unit recording.
These are either intentional extras or duplicate errors (SME Q9).

In [10]:
repeats = merged[(merged["item_quantity"] == 1) & (merged["times_recorded"] > 1)]
print("repeated option groups on qty-1 lines:", len(repeats))
print("of which repeated 5+ times          :", (repeats["times_recorded"] >= 5).sum())
print("fully duplicate option rows         :", options.duplicated().sum())
print()
print(repeats.nlargest(10, "times_recorded").to_string(index=False))

repeated option groups on qty-1 lines: 594
of which repeated 5+ times          : 305
fully duplicate option rows         : 2299

                order_id              lineitem_id option_group_name option_name  times_recorded  item_quantity
5f579c3037ab46b7477b23d3 5f579c42adb0d5c1387b23c9        Vegetables          No              10            1.0
5f57c66f505ee99e507b23ea 5f57c67d37ab4655307b23e4        Vegetables          No              10            1.0
5f590f0137ab46400f7b23f8 5f590f0badb0d51c6a7b23cf        Vegetables          No              10            1.0
5f5913214f5ee92f267b23f1 5f5915d637ab4673257b23d6        Vegetables          No              10            1.0
5f591e52adb0d56b077b2409 5f591efeadb0d529097b241f        Vegetables          No              10            1.0
5f5a30e24f5ee9784c131545 5f5a30e337ab46b22fe9be1e        Vegetables          No              10            1.0
5f5a55f64f5ee9381613152f 5f5a560837ab46141fe9bdf0        Vegetables          No              1

**Conclusion:** options are recorded once per line, and `option_price` is a unit
price (finding 30). Whether the customer is charged once per line or once per
unit can't be determined from the data.

## 8. How much does the option-pricing question matter?

An inner join drops the 28 orphan options (section 9), which can't be tied to a
line item quantity.

In [11]:
opts = options_n.merge(line_quantity, on=LINE_KEY, how="inner")

item_revenue = items_n["item_price"].sum()
options_once = opts["option_price"].sum()
options_per_unit = (opts["option_price"] * opts["item_quantity"]).sum()
difference = options_per_unit - options_once

print(f"item revenue               : ${item_revenue:,.2f}")
print(f"options, once per line     : ${options_once:,.2f}")
print(f"options, x item_quantity   : ${options_per_unit:,.2f}")
print(f"difference                 : ${difference:,.2f} "
      f"({difference / (item_revenue + options_once):.2%} of total revenue)")

item revenue               : $1,778,733.53
options, once per line     : $85,245.14
options, x item_quantity   : $97,700.79
difference                 : $12,455.65 (0.67% of total revenue)


## 9. Data quality checks

### 9.1 Malformed row (finding 8)

In [12]:
malformed = items_n[(items_n["lineitem_id"] == "") | (items_n["item_quantity"] == 0)]
print(malformed.T.to_string())

                                                16529
app_name                                Alltown Fresh
restaurant_id                5e7e35ed902ad5ac017b242b
creation_time_utc    2021-12-07 16:23:30.917000+00:00
order_id                     61af8a82ffd125453e62ae37
user_id                      609d681462e498b356e72a6d
printed_card_number                                  
is_loyalty                                      False
currency                                          USD
lineitem_id                                          
item_category                                        
item_name                                            
item_price                                       4.39
item_quantity                                       0
price_x_qty                                       0.0


### 9.2 Orphan options (finding 29)

Options whose `order_id + lineitem_id` has no line item.

In [13]:
orphans = options.merge(items[LINE_KEY], on=LINE_KEY, how="left", indicator=True)
orphans = orphans[orphans["_merge"] == "left_only"].drop(columns="_merge")

print("orphan option rows   :", len(orphans))
print("distinct line keys   :", orphans[LINE_KEY].drop_duplicates().shape[0])
print("order_id exists in items:", orphans["order_id"].isin(items["order_id"]).sum())
print()
print(orphans.head(10).to_string(index=False))

orphan option rows   : 28
distinct line keys   : 15
order_id exists in items: 0

                order_id              lineitem_id            option_group_name                   option_name option_price option_quantity
65d5d7fc7562bf844306547c 65d5d82c32598edc04095b32          Chilaquiles Options          Add Smokehouse Bacon            2               1
65d5e4ded08987d9b30abb0b 65d5e4de9ec791658a08c16d                 Milk Options               Add Almond Milk            0               1
65d5e4ded08987d9b30abb0b 65d5e4de9ec791658a08c16d             Smoothie Add-ons                   Add Protein            1               1
65d5e4ded08987d9b30abb0b 65d5e4de9ec791658a08c16d             Smoothie Add-ons                     Add Cacao            1               1
65d5ec2aab43396c740b5139 65d5ec4dd5e62eb37906afb7    Breakfast Burrito Options       No Wrap, Make It A Bowl            0               1
65d5ec2aab43396c740b5139 65d5ec4dd5e62eb37906afb7    Breakfast Burrito Options Add Breakfas

### 9.3 Coverage: line items and orders with options (finding 22)

In [14]:
line_keys_with_options = options[LINE_KEY].drop_duplicates()
has_options = items.merge(line_keys_with_options, on=LINE_KEY, how="left", indicator=True)["_merge"] == "both"

orders_with_options = items.loc[has_options.values, "order_id"].nunique()

print(f"line items with options: {has_options.sum():,} of {len(items):,} ({has_options.mean():.0%})")
print(f"orders with options    : {orders_with_options:,} of {items['order_id'].nunique():,} "
      f"({orders_with_options / items['order_id'].nunique():.0%})")

line items with options: 102,697 of 203,519 (50%)
orders with options    : 78,600 of 131,328 (60%)


### 9.4 Apps (findings 12, 13)

In [15]:
by_app = items_n.groupby("app_name").agg(
    rows=("lineitem_id", "size"),
    orders=("order_id", "nunique"),
    customers=("user_id", lambda s: s[s != ""].nunique()),
    item_revenue=("item_price", "sum"),
    first_order=("creation_time_utc", "min"),
    last_order=("creation_time_utc", "max"),
)
print(by_app.to_string())

                               rows  orders  customers  item_revenue                      first_order                       last_order
app_name                                                                                                                              
Alltown Fresh                201423  129511      20026    1763299.87 2020-04-21 18:56:44.244000+00:00 2024-02-21 02:52:35.366000+00:00
Alltown Fresh - DEVELOPMENT     826     741        115       7221.54 2021-03-16 17:47:54.928000+00:00 2024-02-12 14:30:00.133000+00:00
Alltown Neighborhood Perks     1270    1076         34       8212.12 2021-07-06 21:05:30.688000+00:00 2023-03-13 16:20:10.523000+00:00


### 9.5 Heaviest customers (finding 17)

In [16]:
identified = items_n[items_n["user_id"] != ""]
by_user = identified.groupby("user_id").agg(
    line_items=("lineitem_id", "size"),
    orders=("order_id", "nunique"),
    locations=("restaurant_id", "nunique"),
    item_revenue=("item_price", "sum"),
    first_order=("creation_time_utc", "min"),
    last_order=("creation_time_utc", "max"),
)
print(by_user.nlargest(10, "line_items").to_string())
print()
print(by_user[["line_items", "orders", "item_revenue"]].describe(percentiles=[0.5, 0.9, 0.99]))

                          line_items  orders  locations  item_revenue                      first_order                       last_order
user_id                                                                                                                                
5f1b00e5535ee93e0cb768e7        2538    1803         19      27258.73 2020-07-24 14:37:13.836000+00:00 2024-01-22 19:06:23.007000+00:00
5ece77fe902ad501337b23fd        2454    2124         19      23631.93 2020-05-27 14:23:59.294000+00:00 2023-12-28 10:17:54.753000+00:00
609d681462e498b356e72a6d        1133     630          2       4060.57 2021-05-13 19:22:31.961000+00:00 2023-06-21 13:31:51.519000+00:00
5eac88d6902ad598127b240b        1046    1000          6       8175.05 2020-05-01 20:38:00.838000+00:00 2021-06-22 14:05:56.715000+00:00
6282b576c000c6237277bbbd        1010     434          1       7448.28 2022-05-17 13:08:36.444000+00:00 2024-02-20 13:15:38.433000+00:00
5f3c1860505ee9de2a7b23f0         945     743    

### 9.6 Bulk lines (finding 24)

In [17]:
bulk = items_n[items_n["item_quantity"] >= 100]
print("lines with quantity >= 100:", len(bulk))
print(bulk[["app_name", "restaurant_id", "creation_time_utc", "user_id",
            "item_name", "item_price", "item_quantity"]]
      .sort_values("item_quantity", ascending=False).to_string(index=False))

lines with quantity >= 100: 9
     app_name            restaurant_id                creation_time_utc                  user_id                  item_name  item_price  item_quantity
Alltown Fresh 5e7e35f0902ad5ac017b242d 2020-06-03 12:27:22.555000+00:00 5ecf9eda505ee9682b445912          Espresso - Double     1125.00            500
Alltown Fresh 5e7e35f0902ad5ac017b242d 2020-06-03 12:27:22.555000+00:00 5ecf9eda505ee9682b445912    Meet Your Matcha - 12oz     3500.00            500
Alltown Fresh 5e7e35f0902ad5ac017b242d 2020-08-19 12:31:35.028000+00:00 5ee7285f505ee9f7490cf93d              Korean Kimchi     5000.00            500
Alltown Fresh 5fece84aadb0d51509c36f22 2021-10-07 19:37:58.483000+00:00                                  Chili Chicken Bowl     3567.00            300
Alltown Fresh 5fece84aadb0d51509c36f22 2021-10-05 18:59:02.484000+00:00                                             B.L.A.T     2967.00            300
Alltown Fresh 5e7e35f0902ad5ac017b242d 2020-05-28 06:35:07.62300

### 9.7 Inconsistent item names (finding 27)

Names that collide once case and whitespace are normalized. Spelling variants
(e.g. `SARSAPARILA SODA`) need a manual mapping later.

In [18]:
names = items_n[["item_name"]].drop_duplicates()
names["normalized"] = names["item_name"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)

collisions = names.groupby("normalized")["item_name"].agg(list)
collisions = collisions[collisions.str.len() > 1]

print("distinct item_name          :", names["item_name"].nunique())
print("distinct after normalizing  :", names["normalized"].nunique())
print("names with casing variants  :", len(collisions))
print()
print(collisions.head(20).to_string())
print()
print("all-uppercase names:", sorted(names.loc[names["item_name"].str.isupper(), "item_name"]))

distinct item_name          : 432
distinct after normalizing  : 396
names with casing variants  : 36

normalized
avocado toast                                       [Avocado Toast, AVOCADO TOAST]
awake                                                               [Awake, AWAKE]
baja burrito                                          [Baja Burrito, BAJA BURRITO]
buffalo chicken wrap                  [Buffalo Chicken Wrap, BUFFALO CHICKEN WRAP]
carnitas torta                                    [Carnitas Torta, CARNITAS TORTA]
chicken teriyaki bowl               [Chicken Teriyaki Bowl, CHICKEN TERIYAKI BOWL]
chili chicken                                       [Chili Chicken, CHILI CHICKEN]
chipotle turkey                                 [Chipotle Turkey, CHIPOTLE TURKEY]
chula vista                                             [CHULA VISTA, Chula Vista]
eggplant sugo bowl                        [Eggplant Sugo Bowl, EGGPLANT SUGO BOWL]
falafel wrap                                          [Fa

## 10. Line revenue with the working formula

```
line revenue = item_price + Σ (option_price × item_quantity)
```

Options are summed per line first, then **left-joined**, so the 50% of line items
without options keep their revenue.

In [19]:
option_price_per_line = (
    options_n.groupby(LINE_KEY, as_index=False)["option_price"].sum()
    .rename(columns={"option_price": "option_price_sum"})
)

lines = items_n.merge(option_price_per_line, on=LINE_KEY, how="left", validate="one_to_one")
lines["option_price_sum"] = lines["option_price_sum"].fillna(0)
lines["line_revenue"] = lines["item_price"] + lines["option_price_sum"] * lines["item_quantity"]

assert len(lines) == len(items_n), "left join changed the row count"

print(f"line items        : {len(lines):,}")
print(f"total revenue     : ${lines['line_revenue'].sum():,.2f}  (all years, before exclusions)")

line items        : 203,519
total revenue     : $1,876,434.32  (all years, before exclusions)


## 11. The 2023 population

SME confirmed a 2023-only scope. Timestamps are UTC, so the business day depends
on the time zone (SME Q10). Both boundaries are measured: orders placed late on
Dec 31 Eastern time fall on Jan 1 in UTC.

In [20]:
lines["order_date_utc"] = lines["creation_time_utc"].dt.date
lines["order_date_local"] = lines["creation_time_utc"].dt.tz_convert(BUSINESS_TZ).dt.date

def scope_summary(df: pd.DataFrame) -> dict:
    return {
        "line_items": len(df),
        "orders": df["order_id"].nunique(),
        "customers": df.loc[df["user_id"] != "", "user_id"].nunique(),
        "guest_lines": (df["user_id"] == "").sum(),
        "locations": df["restaurant_id"].nunique(),
        "revenue": round(df["line_revenue"].sum(), 2),
    }

in_2023_utc = pd.to_datetime(lines["order_date_utc"]).dt.year == 2023
in_2023_local = pd.to_datetime(lines["order_date_local"]).dt.year == 2023

summary = pd.DataFrame.from_dict({
    "all_years": scope_summary(lines),
    "2023_utc": scope_summary(lines[in_2023_utc]),
    f"2023_{BUSINESS_TZ}": scope_summary(lines[in_2023_local]),
}, orient="index")
print(summary.to_string())
print()
print("rows whose year differs between UTC and local:", (in_2023_utc != in_2023_local).sum())

                       line_items  orders  customers  guest_lines  locations     revenue
all_years                  203519  131328      20174        17808         28  1876434.32
2023_utc                    80664   52640      10604         6395         21   752769.30
2023_America/New_York       80665   52641      10604         6395         21   752776.30

rows whose year differs between UTC and local: 1


### Orders per year and date_dim coverage

In [21]:
order_years = lines.drop_duplicates("order_id")["creation_time_utc"].dt.tz_convert(BUSINESS_TZ).dt.year
print("orders per year (local):")
print(order_years.value_counts().sort_index().to_string())

local_2023_dates = set(pd.to_datetime(lines.loc[in_2023_local, "order_date_local"]))
dim_dates = set(dates_n["date_key"])
print()
print("2023 order dates missing from date_dim:", len(local_2023_dates - dim_dates))
print("date_dim days with no orders          :", len(dim_dates - local_2023_dates))

orders per year (local):
creation_time_utc
2020     8186
2021    26790
2022    37490
2023    52641
2024     6221

2023 order dates missing from date_dim: 0
date_dim days with no orders          : 0


### Excluding test data

SME confirmed (2026-10-04) that `Alltown Fresh - DEVELOPMENT` rows are test data
and are excluded from metrics (finding 12, SME Q5). The rows are flagged rather
than deleted, so the pipeline keeps them in raw and cleaned layers and filters at
the metrics layer.

In [22]:
lines["is_test_data"] = lines["app_name"] == "Alltown Fresh - DEVELOPMENT"

in_scope = in_2023_local & ~lines["is_test_data"]
print(pd.DataFrame.from_dict({
    "2023_local": scope_summary(lines[in_2023_local]),
    "2023_local_excl_test": scope_summary(lines[in_scope]),
}, orient="index").to_string())

test_locations = set(lines.loc[lines["is_test_data"], "restaurant_id"])
real_locations = set(lines.loc[~lines["is_test_data"], "restaurant_id"])
print()
print("restaurant_ids used only by test data:", sorted(test_locations - real_locations))

                      line_items  orders  customers  guest_lines  locations    revenue
2023_local                 80665   52641      10604         6395         21  752776.30
2023_local_excl_test       79965   52015      10513         6132         20  746223.86

restaurant_ids used only by test data: ['6050e76361e498ca740bba6f']


### 2023 impact of the open findings

The open findings, recounted for the in-scope population (2023, local time, test
data excluded). Several issues found across all years barely occur in 2023.

In [23]:
s = lines[in_scope]
top_users = by_user.nlargest(2, "line_items").index

scoped_counts = option_counts.merge(s[[*LINE_KEY, "item_quantity"]], on=LINE_KEY, how="inner")
scoped_names = s[["item_name"]].drop_duplicates()
scoped_names["normalized"] = scoped_names["item_name"].str.strip().str.lower()

print(f"guest lines (blank user_id)        : {(s['user_id'] == '').sum():,} ({(s['user_id'] == '').mean():.1%})")
print(f"malformed row                      : {s['lineitem_id'].eq('').sum()}")
print(f"repeated options on qty-1 lines    : {((scoped_counts['item_quantity'] == 1) & (scoped_counts['times_recorded'] > 1)).sum()}")
print(f"Neighborhood Perks rows            : {(s['app_name'] == 'Alltown Neighborhood Perks').sum():,}")
print(f"rows from top-2 user_ids           : {s['user_id'].isin(top_users).sum():,} "
      f"{s.loc[s['user_id'].isin(top_users), 'user_id'].value_counts().to_dict()}")
print(f"quantity >= 100 rows               : {(s['item_quantity'] >= 100).sum():,} (max quantity {s['item_quantity'].max()})")
print(f"max item_price                     : ${s['item_price'].max():,.2f}")
print(f"$0 item_price rows                 : {(s['item_price'] == 0).sum():,}")
print(f"item names / after lowercasing     : {scoped_names['item_name'].nunique()} / {scoped_names['normalized'].nunique()}")
print(f"loyalty lines (TRUE / FALSE)       : {s['is_loyalty'].sum():,} / {(~s['is_loyalty']).sum():,}")

guest lines (blank user_id)        : 6,132 (7.7%)
malformed row                      : 0
repeated options on qty-1 lines    : 0
Neighborhood Perks rows            : 4
rows from top-2 user_ids           : 1,136 {'5ece77fe902ad501337b23fd': 1090, '5f1b00e5535ee93e0cb768e7': 46}
quantity >= 100 rows               : 0 (max quantity 27)
max item_price                     : $269.73
$0 item_price rows                 : 1
item names / after lowercasing     : 122 / 120
loyalty lines (TRUE / FALSE)       : 38,932 / 41,033


### Corrupted item categories (finding 34)

Some `item_category` values contain a pasted admin URL in the middle of the word,
for example `Drip Chttps://www.opendining.net/...#offee`. The URL always starts with
`http` and ends with `#`, so removing everything from `http` to the first `#`
should give back the real category. The check below confirms that every repaired
value is a category that already exists in clean rows.

In [24]:
URL_FRAGMENT = r"https?://\S*?#"   # pasted admin link: from "http" up to the first "#"

corrupted = s["item_category"].str.contains(r"https?://", regex=True)
repaired = s.loc[corrupted, "item_category"].str.replace(URL_FRAGMENT, "", regex=True)
clean_categories = set(s.loc[~corrupted, "item_category"])

print(f"corrupted rows in scope: {corrupted.sum():,} of {len(s):,}")
print(f"distinct corrupted values: {s.loc[corrupted, 'item_category'].nunique()}")
print("repaired to:", repaired.value_counts().to_dict())
print("every repaired value is an existing clean category:", set(repaired) <= clean_categories)

corrupted rows in scope: 96 of 79,965
distinct corrupted values: 2
repaired to: {'BBQ Plates': 70, 'Drip Coffee': 26}
every repaired value is an existing clean category: True


**Conclusion:** the corruption is mechanical and always reversible. The pipeline
removes the URL fragment in silver and flags the repaired rows (`is_category_repaired`).

### Loyalty status changes over time (finding 35)

`is_loyalty` is recorded on every order. If it were a fixed property of the customer,
each customer would have only one value. The check below counts customers whose
orders have both values, and in which order. Guest orders and the confirmed
non-customer account (SME Q11) are excluded, as in customer metrics.

In [25]:
NON_CUSTOMER_IDS = ["5ece77fe902ad501337b23fd"]   # SME confirmed faulty account (Q11)

customer_orders = (
    s[(s["user_id"] != "") & ~s["user_id"].isin(NON_CUSTOMER_IDS)]
    .drop_duplicates("order_id")                      # is_loyalty is identical on every line of an order
    .sort_values("creation_time_utc")
)
loyalty = customer_orders.groupby("user_id")["is_loyalty"]
mixed = loyalty.nunique() > 1
started_non_loyal = ~loyalty.first().astype(bool)
switches = loyalty.apply(lambda flags: int((flags != flags.shift()).sum()) - 1)

print(f"customers in scope: {loyalty.ngroups:,}")
print(f"with both loyalty and non-loyalty orders: {mixed.sum():,} ({mixed.mean():.1%})")
print(f"  of those, first order was non-loyalty (joined later): {(mixed & started_non_loyal).sum():,}")
print(f"  most switches by one customer: {switches.max()}")

customers in scope: 10,512
with both loyalty and non-loyalty orders: 1,913 (18.2%)
  of those, first order was non-loyalty (joined later): 1,808
  most switches by one customer: 9


**Conclusion:** loyalty is not a fixed customer attribute. The data model keeps
`is_loyalty` on every order and records each customer's loyalty status as of each day
in the daily snapshot, so loyalty comparisons use the status at the time.

## 12. Summary

| Section | Result | Finding |
|---|---|---|
| 4 | Blank `printed_card_number` ⇔ `is_loyalty` FALSE, zero exceptions | 7 |
| 4 | No negative `option_price`; discounts not represented as documented | 23 |
| 6 | `item_price` is the line total (0 of 13,650 rows match unit price) | 28 |
| 7 | Options recorded once per line; `option_price` is a unit price | 30 |
| 7 | Repeated identical options on qty-1 lines | 9 |
| 8 | Once-per-line vs × quantity differs by 0.67% of revenue | SME Q3 |
| 9 | Malformed row, orphan options, DEVELOPMENT app, heavy users, bulk lines, name variants | 8, 29, 12, 17, 24, 27 |
| 10 | Working formula: `item_price + Σ(option_price × item_quantity)`, left join | |
| 11 | 2023 population measured under UTC and local time | SME Q10 |
| 11 | DEVELOPMENT rows excluded as test data (SME confirmed); 2023 in scope: 52,015 orders, 20 locations | 12 |
| 11 | `item_category` corrupted by pasted admin URLs; removing the URL fragment always gives an existing category | 34 |
| 11 | Customers' loyalty status changes over time; modelled per order and per day | 35 |

Numbers in the report (`docs/02_data_analysis/Step2_Initial_Data_Analysis.md`)
must match the outputs above.